In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture
from collections import deque, defaultdict

# CAE

In [2]:
train = pd.read_csv('../../../train_CADE_cod.csv')
val = pd.read_csv('../../../val_CADE_cod.csv')
test = pd.read_csv('../../../test_CADE_cod.csv')

In [3]:
classes_out = ['Infilteration','DoS attacks-SlowHTTPTest']
train = train.drop(index=train[train['Label'].isin(classes_out)].index)
val = val.drop(index=val[val['Label'].isin(classes_out)].index)
test = test.drop(index=test[test['Label'].isin(classes_out)].index)

# CAE CADE (margin 1 e lambda 1.0)

In [4]:
def pick_pair(embeddings, labels, pick_embeddings, pick_labels):
    """
    Gera todos os pares entre embeddings e pick_embeddings,
    com labels_pair indicando se os pares são positivos (0) ou negativos (1).
    """
    N, D = embeddings.shape
    M = pick_embeddings.shape[0]

    # Expande para todas as combinações possíveis
    A = embeddings.unsqueeze(0).expand(M, N, D).reshape(M * N, D)
    B = pick_embeddings.unsqueeze(1).expand(M, N, D).reshape(M * N, D)

    labels_exp = labels.unsqueeze(0).expand(M, N)
    pick_labels_exp = pick_labels.unsqueeze(1).expand(M, N)
    labels_pair = (labels_exp != pick_labels_exp).reshape(-1).float()

    return A, B, labels_pair

def ContrastiveLoss(input1, input2, y, margin=1.0):
    # y = 0 quando input1 e input2 são da mesma classe e y = 1 caso contrário
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff,2),1)
    dist = torch.sqrt(dist_sq + 1e-10) # Soma 1e-10 pois se o sqrt der 0 o gradiente da NaN
    mdist = margin - dist
    mdist = F.relu(mdist)
    loss = ((1-y) * dist * dist) + (y * mdist * mdist)
    return torch.mean(loss)

def AEContrastive(input, output, embeddings, labels, pick_encoded, pick_labels, margin=1.0, lambda1=1.0):
    mseloss = nn.MSELoss()
    ae_loss = mseloss(input,output)

    A, B, labels_pair = pick_pair(embeddings, labels, pick_encoded, pick_labels)
    contrastive_loss = ContrastiveLoss(A, B, labels_pair, margin=margin)
    return ae_loss + contrastive_loss * lambda1, ae_loss, contrastive_loss

In [5]:
class contrastive_ae(nn.Module):
    def __init__(self, input_neurons, neurons):
        super().__init__()
        self.encoder0 = nn.Linear(input_neurons,64)
        self.encoder1 = nn.Linear(64,32)
        self.encoder2 = nn.Linear(32,16)
        self.encoder3 = nn.Linear(16,neurons)

        self.decoder0 = nn.Linear(neurons,16)
        self.decoder1 = nn.Linear(16,32)
        self.decoder2 = nn.Linear(32,64)
        self.decoder3 = nn.Linear(64,input_neurons)

        self.activation0 = nn.ReLU()
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        encoded = X
        X = self.activation0(self.decoder0(X))
        X = self.activation0(self.decoder1(X))
        X = self.activation0(self.decoder2(X))
        X = self.decoder3(X)


        return X, encoded

In [6]:
class encoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder0 = list(model.children())[0]
        self.encoder1 = list(model.children())[1]
        self.encoder2 = list(model.children())[2]
        self.encoder3 = list(model.children())[3]
        self.activation0 = list(model.children())[8]
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        return X

In [7]:
from torch.utils.data import Dataset, DataLoader,TensorDataset
import random
def build_class_reference_batches(X_train, y_train, batch_size=512):
    classes = torch.unique(y_train).tolist()
    n_classes = len(classes)
    total_samples = len(X_train)
    dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    

    print(classes)

    # Mapeia cada classe para seus índices
    class_to_indices = {
        int(cls.item()): (y_train == cls).nonzero(as_tuple=True)[0].tolist()
        for cls in torch.unique(y_train)
    }

    reference_list = []
    total_batches = len(train_loader)

    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_samples)
        batch_indices = set(range(start, end))

        class_samples = []
        for cls in classes:
            candidates = [i for i in class_to_indices[cls] if i not in batch_indices]
            if not candidates:
                raise ValueError(f"Classe {cls} não possui amostras fora do batch {batch_idx}")
            chosen = candidates[torch.randint(len(candidates), (1,)).item()]
            class_samples.append(chosen)

        reference_list.append(class_samples)


    return train_loader, reference_list

In [8]:
def normalize(t):
    t_norm = (t - t.mean()) / t.std()
    return t_norm


In [9]:
train['Label'] = train['Label'].replace('FTP-BruteForce','BruteForce')
train['Label'] = train['Label'].replace('SSH-Bruteforce','BruteForce')
train['Label'] = train['Label'].replace('DoS attacks-GoldenEye','DoS')
# train['Label'] = train['Label'].replace('DoS attacks-Slowloris','DoS')
## train['Label'] = train['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Hulk','DoS')
train['Label'] = train['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-HOIC','DDoS')
train['Label'] = train['Label'].replace('Brute Force -Web','Web')
train['Label'] = train['Label'].replace('Brute Force -XSS','Web')
train['Label'] = train['Label'].replace('SQL Injection','Web')


val['Label'] = val['Label'].replace('FTP-BruteForce','BruteForce')
val['Label'] = val['Label'].replace('SSH-Bruteforce','BruteForce')
val['Label'] = val['Label'].replace('DoS attacks-GoldenEye','DoS')
# val['Label'] = val['Label'].replace('DoS attacks-Slowloris','DoS')
## val['Label'] = val['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Hulk','DoS')
val['Label'] = val['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-HOIC','DDoS')
val['Label'] = val['Label'].replace('Brute Force -Web','Web')
val['Label'] = val['Label'].replace('Brute Force -XSS','Web')
val['Label'] = val['Label'].replace('SQL Injection','Web')


test['Label'] = test['Label'].replace('FTP-BruteForce','BruteForce')
test['Label'] = test['Label'].replace('SSH-Bruteforce','BruteForce')
test['Label'] = test['Label'].replace('DoS attacks-GoldenEye','DoS')
# test['Label'] = test['Label'].replace('DoS attacks-Slowloris','DoS')
## test['Label'] = test['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Hulk','DoS')
test['Label'] = test['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-HOIC','DDoS')
test['Label'] = test['Label'].replace('Brute Force -Web','Web')
test['Label'] = test['Label'].replace('Brute Force -XSS','Web')
test['Label'] = test['Label'].replace('SQL Injection','Web')

In [10]:
labels_str = [i for i in train['Label'].value_counts().index]
print(labels_str)

for i, l in enumerate(labels_str):
    train['Label'] = train['Label'].replace(l, i)
    val['Label'] = val['Label'].replace(l, i)
    test['Label'] = test['Label'].replace(l, i)

['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'DoS attacks-Slowloris', 'Web']


C:\Users\linco\AppData\Local\Temp\ipykernel_14012\2307520911.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Label'] = train['Label'].replace(l, i)
C:\Users\linco\AppData\Local\Temp\ipykernel_14012\2307520911.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['Label'] = val['Label'].replace(l, i)
C:\Users\linco\AppData\Local\Temp\ipykernel_14012\2307520911.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result

In [11]:
array_hidden_classes = [[5]]
filenames = ['5']
# array_hidden_classes = [[0]]
# filenames = ['0']
total_classes = len(labels_str)
for i in range(len(array_hidden_classes)):
    filename = f'{filenames[i]}_hidden'
    print(filename)
    hidden_classes = array_hidden_classes[i] #Classes ocultas do treinamento

    hidden_classes_index = list(train[train['Label'].isin(hidden_classes)].index)

    train_hidden = train.loc[hidden_classes_index]

    train_not_hidden = train.drop(hidden_classes_index)

    X_train = train_not_hidden.drop(columns=['Label'])
    y_train = train_not_hidden['Label']
    X_val = val.drop(columns=['Label'])
    y_val = val['Label'].values
    X_test = test.drop(columns=['Label'])
    y_test = test['Label'].values

    torch.manual_seed(123)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_loader, pick_list = build_class_reference_batches(X_train,y_train, batch_size=512)
    pick_samples = []
    pick_labels = []
    for l in pick_list:
        pick_samples.append(X_train[l])
        pick_labels.append(y_train[l])

    pick_samples = torch.stack(pick_samples)    
    pick_labels = torch.stack(pick_labels)

    print('pick completo')

    neurons = total_classes - len(hidden_classes)
    print(f'{X_train.shape[1]} {neurons}')
    model = contrastive_ae(input_neurons=X_train.shape[1],neurons=neurons)
    model.to(device)

    learning_rate = 0.0001
    opt = optim.Adam(model.parameters(),lr=learning_rate)
    margin = 10 # Parametro de afastamento das classes
    cae_lambda_1 = 0.1 # Parametro de influencia na distancia das classes
    
    
    for epoch in range(250):
        running_loss = 0
        running_ael = 0
        running_cl = 0
        contador = 0
        for data in train_loader:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            ps = pick_samples[contador]
            pl = pick_labels[contador]
            ps = ps.to(device)
            pl = pl.to(device)
            contador += 1
            #print(f'{contador}/{len(train_loader)}')
            opt.zero_grad()
            outputs, encoded = model(inputs)
            _, pick_encoded = model(ps)
            tam_pe = pick_encoded.shape[0]
            enc_total = torch.cat((encoded,pick_encoded), dim=0)
            enc_total_norm = normalize(enc_total)
            encoded_norm = enc_total_norm[:-tam_pe]
            pick_encoded_norm = enc_total_norm[-tam_pe:]
            loss, ael, cl = AEContrastive(input=inputs, output=outputs, embeddings = encoded_norm, labels=labels, pick_encoded = pick_encoded_norm, pick_labels = pl, margin=margin, lambda1=cae_lambda_1)
            #loss = dummyMSELoss(inputs,outputs)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            running_ael += ael.item()
            running_cl += cl.item()
            #print(f'loss = {loss.item()} batch_size = {size}')
        print(f'filename: {filename} Epoch {epoch+1} loss:{running_loss/len(train_loader)} ael:{running_ael/len(train_loader)} cl:{running_cl/len(train_loader)}')

    model_e = encoder(model=model)
    model_e.to(device)

    model_e.eval()

    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_encoded = model_e(X_train.to(device))

    train_encoded = train_encoded.detach().cpu().numpy()

    train_encoded_df = pd.DataFrame(train_encoded)
    train_encoded_df['Label'] = y_train.detach().cpu().numpy().astype(int)
    display(train_encoded_df)

    # score = davies_bouldin_score(train_encoded_df.drop(columns=['Label']), train_encoded_df['Label'])
    # print("Davies-Bouldin Score:", score)

    X_val = torch.tensor(np.array(X_val), dtype=torch.float)

    val_encoded = model_e(X_val.to(device))

    val_encoded = val_encoded.detach().cpu().numpy()

    val_encoded_df = pd.DataFrame(val_encoded)

    val_encoded_df['Label'] = y_val

    display(val_encoded_df)

    X_test = torch.tensor(np.array(X_test), dtype=torch.float)

    test_encoded = model_e(X_test.to(device))

    test_encoded = test_encoded.detach().cpu().numpy()

    test_encoded_df = pd.DataFrame(test_encoded)

    test_encoded_df['Label'] = y_test

    display(test_encoded_df)

    train_encoded_df.to_csv(f'train_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)
    val_encoded_df.to_csv(f'val_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)
    test_encoded_df.to_csv(f'test_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)

5_hidden
cuda
[0.0, 1.0, 2.0, 3.0, 4.0, 6.0]
pick completo
82 6
filename: 5_hidden Epoch 1 loss:3.678728513112331 ael:0.021824234148690322 cl:36.569042164827756
filename: 5_hidden Epoch 2 loss:2.9464056456521814 ael:0.015231017770177128 cl:29.31174582050123
filename: 5_hidden Epoch 3 loss:2.818254288699074 ael:0.014783351872807287 cl:28.034708915271416
filename: 5_hidden Epoch 4 loss:2.762050104541642 ael:0.014049225022878786 cl:27.480008336578543
filename: 5_hidden Epoch 5 loss:2.73553748082514 ael:0.013786251024175394 cl:27.217511828783618
filename: 5_hidden Epoch 6 loss:2.7146105742154223 ael:0.0133460031453843 cl:27.01264524424509
filename: 5_hidden Epoch 7 loss:2.699550152708201 ael:0.012727512054515198 cl:26.868225946989316
filename: 5_hidden Epoch 8 loss:2.6821926897854005 ael:0.012178411395154446 cl:26.700142292070755
filename: 5_hidden Epoch 9 loss:2.6686739003578803 ael:0.011622668145269074 cl:26.570511825704727
filename: 5_hidden Epoch 10 loss:2.6569065623708523 ael:0.011125

C:\Users\linco\AppData\Local\Temp\ipykernel_14012\2916349630.py:91: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train = torch.tensor(np.array(X_train), dtype=torch.float)
C:\Users\linco\AppData\Local\Temp\ipykernel_14012\2916349630.py:92: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train), dtype=torch.float)


,0,1,2,3,4,5,Label
0,0.014890,-0.023060,0.009290,-0.022377,-0.003735,-0.028105,1
1,-0.031908,-0.014282,-0.019942,-0.001233,-0.020599,-0.029237,0
2,0.009689,0.027645,-0.021565,0.015003,0.005647,0.009302,2
3,0.019207,-0.020570,0.022424,-0.013071,0.001536,-0.039974,1
4,0.001443,-0.024458,-0.068531,-0.067295,-0.031014,0.010318,4
...,...,...,...,...,...,...,...
2072817,-0.031585,-0.014109,-0.019665,-0.001096,-0.020368,-0.029185,0
2072818,0.009687,0.027643,-0.021563,0.015003,0.005646,0.009298,2
2072819,0.009686,0.027641,-0.021560,0.015004,0.005646,0.009295,2
2072820,-0.032083,-0.014376,-0.020092,-0.001308,-0.020724,-0.029265,0


,0,1,2,3,4,5,Label
0,-0.030841,-0.013628,-0.019040,-0.001387,-0.019843,-0.029518,0
1,0.009693,0.027653,-0.021575,0.015002,0.005648,0.009314,2
2,0.014322,-0.022052,0.010054,-0.022268,-0.003364,-0.029962,1
3,-0.045550,-0.030414,0.023386,-0.021022,-0.013828,0.040278,3
4,0.013015,-0.022285,0.009108,-0.023939,-0.004070,-0.030613,1
...,...,...,...,...,...,...,...
519951,0.013826,-0.021345,0.008977,-0.021608,-0.003549,-0.029293,1
519952,0.007435,-0.023749,-0.067271,-0.071060,-0.028780,0.012486,4
519953,-0.031668,-0.014153,-0.019736,-0.001131,-0.020427,-0.029198,0
519954,0.012639,-0.022591,0.007422,-0.020660,-0.004921,-0.029739,1


,0,1,2,3,4,5,Label
0,-0.031874,-0.014264,-0.019913,-0.001219,-0.020575,-0.029232,0
1,-0.030547,-0.013678,-0.018319,-0.001149,-0.019575,-0.029863,0
2,0.013995,-0.019731,0.009079,-0.019966,-0.002899,-0.029110,1
3,0.014012,-0.024849,0.012140,-0.023239,-0.004085,-0.033946,1
4,-0.030941,-0.013096,-0.018402,-0.001554,-0.019340,-0.029018,0
...,...,...,...,...,...,...,...
649942,0.012307,-0.022752,0.008320,-0.024228,-0.004692,-0.030806,1
649943,-0.033238,-0.014046,-0.019691,-0.001944,-0.020662,-0.029706,0
649944,-0.032493,-0.013759,-0.019077,-0.001708,-0.020184,-0.029627,0
649945,-0.041207,-0.027096,0.021783,-0.018802,-0.012160,0.037452,3
